# nmtc-application-builder — Quickstart

**Week 1 of 4: Foundation & Pipeline Intelligence**

This notebook demonstrates the complete Week 1 workflow:
1. Load a sample CDE profile
2. Load a sample pipeline (20 projects)
3. Run `app.analyze()`
4. Print the rich summary
5. Inspect the readiness score

All calls work offline — external APIs fall back to sample data automatically.

In [ ]:
import sys
sys.path.insert(0, '..')  # add repo root to path

from nmtcapp.core.application import Application
from nmtcapp.core.cde import CDEProfile
from nmtcapp.core.pipeline import Pipeline

## Step 1 — Load a Sample CDE Profile

In [ ]:
cde = CDEProfile.sample()

print(f"CDE:               {cde.name}")
print(f"CDE ID:            {cde.cde_id}")
print(f"Certified:         {cde.certification_date}")
print(f"Target Markets:    {', '.join(cde.target_markets)}")
print(f"Prior Awards:      {len(cde.prior_awards)} rounds, ${cde.total_prior_allocation():,.0f} total")
print(f"\nMission: {cde.mission[:120]}...")

## Step 2 — Load a Sample Pipeline (20 projects)

In [ ]:
pipeline = Pipeline.sample(n=20)

print(f"Pipeline: {len(pipeline)} projects")
print(f"Total QEI: ${sum(p.qei_request for p in pipeline):,.0f}")
print()

df = pipeline.to_dataframe()
display(df[['project_name', 'state', 'sector', 'qei_request', 'distress_level', 'expected_jobs_created']].head(10))

In [ ]:
# Pipeline sector and state distribution
print("Sector breakdown:")
print(df['sector'].value_counts().to_string())
print()
print("State breakdown:")
print(df['state'].value_counts().to_string())

## Step 3 — Create Application and Run analyze()

In [ ]:
app = Application(
    cde=cde,
    requested_allocation=65_000_000,
    application_round="CY2025",
)
app.add_pipeline(pipeline)

print("Running analysis...")
analysis = app.analyze()
print("Done.")

## Step 4 — Print the Rich Summary

In [ ]:
analysis.summary()

## Step 5 — Inspect the Readiness Score

In [ ]:
score = analysis.readiness_score
print(score.summary())

In [ ]:
# Component scores as a DataFrame
import pandas as pd

components_df = pd.DataFrame.from_dict(
    score.component_scores, orient='index', columns=['Score (0-100)']
)
components_df.index.name = 'Component'
display(components_df.sort_values('Score (0-100)', ascending=False))

## Bonus — Distress Concentration Deep Dive

In [ ]:
distress = analysis.distress_analysis

print("Distress Concentration Summary")
print("=" * 40)
print(f"  Deep:              {distress['pct_deep']:.1%}")
print(f"  Severe (excl. deep): {distress['pct_severe_excluding_deep']:.1%}")
print(f"  Deep + Severe:     {distress['pct_deep_or_severe']:.1%}")
print(f"  Standard LIC:      {distress['pct_lic']:.1%}")
print(f"  Native Area:       {distress['pct_native_area']:.1%}")
print(f"  High Mig. Rural:   {distress['pct_high_migration_rural']:.1%}")
print()
print(f"  Meets min threshold (50%):    {distress['meets_min_threshold']}")
print(f"  Meets target threshold (75%): {distress['meets_target_threshold']}")

## Bonus — Loading from CSV

You can also load your pipeline from a CSV file:

In [ ]:
# Export the sample pipeline to CSV, then reload it
import tempfile, os

with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False) as f:
    pipeline.to_dataframe().to_csv(f.name, index=False)
    csv_path = f.name

reloaded = Pipeline.from_csv(csv_path)
print(f"Reloaded {len(reloaded)} projects from CSV")
os.unlink(csv_path)

In [ ]:
# Week 2: Generate the full application package
import os

paths = app.generate(
    output_dir="../examples/sample_output",
    formats=["markdown", "word", "excel", "pdf"],
)

print("Generated files:")
for fmt, path in paths.items():
    size_kb = os.path.getsize(path) // 1024
    print(f"  {fmt:10s} → {os.path.basename(path)}  ({size_kb} KB)")

## Week 2 Complete — What's Next

Week 2 deliverables are now complete:
- ✅ **Section generators** A–E with narrative, table, and list subsections
- ✅ **Table builders** — pipeline, distress, geographic, impact, investor, track record
- ✅ **Word output** —  with cover page, sections, appendices
- ✅ **Excel output** — 7-sheet workbook with conditional formatting and charts
- ✅ **PDF output** — board-ready PDF via ReportLab
- ✅ **Markdown output** — version-control-friendly  file

See  for the complete output generation demo.

Week 3 will add: win probability modeling, optimizer, HMDA integration, and visualizations.

See the [GitHub repo](https://github.com/Jaypatel1511/nmtc-application-builder) for the full roadmap.